# EDA — `bronze.trust_prices_yf`

The price source for every trust. This is the table the answer depends on, so it gets
the most attention.

**This notebook decides nothing.** It measures what is in the data and hands the
decisions to the Silver spec. Bronze is all STRING, so every numeric check casts
explicitly — which doubles as documentation of what Silver has to cast.

## 1. Shape

In [0]:
%sql
SELECT COUNT(*)               AS rows,
       COUNT(DISTINCT symbol) AS symbols,
       MIN(`Date`)            AS first_bar,
       MAX(`Date`)            AS last_bar
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf;

**35,121 rows, 100 symbols, 1967-12 to the current month.** The history is far deeper
than the CSV's 2011 floor, which is the main reason the source was changed.

## 2. Can the month key just be a substring?

The CSV needed a `day <= 3` rule because its date labels mixed month-end with
next-month-first. Check whether Yahoo has the same problem.

In [0]:
%sql
SELECT SUBSTRING(`Date`, 9, 2) AS day_of_month,
       COUNT(*)                AS rows
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
GROUP BY day_of_month
ORDER BY rows DESC;

In [0]:
%sql
-- If two rows ever shared a (symbol, month) the key would need a tie-break rule.
SELECT COUNT(*) AS colliding_groups
FROM (
  SELECT symbol, SUBSTRING(`Date`, 1, 7) AS month_key
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  GROUP BY symbol, month_key
  HAVING COUNT(*) > 1
);

**35,120 of 35,121 bars fall on day 01**, the exception being `ADIG.L` at 2026-03-27 —
a single-row stub. And **0 colliding groups**.

So the month key is `SUBSTRING(Date, 1, 7)` with no tie-break rule. The CSV's `day <= 3`
rule dies with the CSV. **Settled.**

## 3. Unusable prices

In [0]:
%sql
SELECT SUM(CASE WHEN `Close` IS NULL THEN 1 ELSE 0 END)              AS null_close,
       SUM(CASE WHEN CAST(`Close` AS DOUBLE) <= 0 THEN 1 ELSE 0 END) AS non_positive_close,
       COUNT(*)                                                      AS total_rows
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf;

In [0]:
%sql
SELECT symbol, `Date`, `Close`
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
WHERE CAST(`Close` AS DOUBLE) <= 0;

**No nulls, and exactly one zero: `PCFT.L` at 2019-11-01.**

Worth noticing that this is the *same row* the CSV archive has as its known-bad value.
The CSV did not invent the defect — it inherited it from Yahoo.

A zero price breaks a return twice: once as the denominator of that month's return, and
once as the numerator of the next. **Open for Silver.**

## 4. Where each series starts and stops

Two things matter: whether the most recent month is complete, and how many trusts have
enough history for each horizon.

In [0]:
%sql
SELECT last_month, COUNT(*) AS symbols
FROM (
  SELECT symbol, SUBSTRING(MAX(`Date`), 1, 7) AS last_month
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  GROUP BY symbol
)
GROUP BY last_month
ORDER BY last_month DESC;

In [0]:
%sql
-- How much history each symbol actually has. These become the horizon denominators.
SELECT CASE WHEN years >= 15 THEN '15y or more'
            WHEN years >= 10 THEN '10 to 15y'
            WHEN years >= 5  THEN '5 to 10y'
            WHEN years >= 3  THEN '3 to 5y'
            ELSE 'under 3y (stub)' END AS history,
       COUNT(*) AS symbols
FROM (
  SELECT symbol,
         MONTHS_BETWEEN(MAX(`Date`), MIN(`Date`)) / 12 AS years
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  GROUP BY symbol
)
GROUP BY history
ORDER BY symbols DESC;

**97 symbols stop at the current month, 2 at 2026-08, 1 at 2026-03.**

The current month is still being traded, so its bar is a **partial month** — not
comparable with the complete months before it. **Open for Silver.**

History depth: **75 at 15y+, 10 at 10–15y, 10 at 5–10y, 1 at 3–5y, 4 stubs.** The four
stubs (`ADIG`, `BSIF`, `EOT`, `MNTN`) carry one or two bars each. **Open for Silver.**

## 5. Stock splits

A split multiplies the share count and divides the price. If Yahoo has already applied it
backwards through the history, `Close` stays continuous and nothing needs doing. If not,
every split looks like a crash.

In [0]:
%sql
SELECT COUNT(*)               AS split_events,
       COUNT(DISTINCT symbol) AS symbols
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
WHERE CAST(Stock_Splits AS DOUBLE) <> 0;

In [0]:
%sql
-- BNKR split 10-for-1 in March 2021. If Close is split-adjusted it runs smoothly through.
SELECT SUBSTRING(`Date`, 1, 7)          AS month,
       ROUND(CAST(`Close` AS DOUBLE), 2) AS close,
       Stock_Splits
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
WHERE symbol = 'BNKR.L' AND `Date` >= '2021-01-01' AND `Date` < '2021-06-01'
ORDER BY `Date`;

**50 split events across 39 symbols**, and `BNKR` runs 105.5 → 106.35 → 110.8 → 114.2
straight through its 10-for-1. `Close` is already split-adjusted, so splits need no rule
of their own. **Settled.**

## 6. Rows recorded at the wrong scale

The biggest finding in this notebook. Some months are reported in a different unit from
their neighbours — `CGT` alternates between roughly 485 and 4,870.

To detect it, compare each month against the **median of its 13-month neighbourhood** (6
before, 6 after, same symbol). The median is used rather than the average precisely
because a handful of extreme values cannot drag it: for the `CGT` window below the
average is about 1,571 and the median is 484.75.

The median is computed over each symbol's **full** history, then the window is filtered
afterwards — otherwise the earliest months are judged against a truncated half-window.

In [0]:
%sql
-- The defect, in one trust. Three months sit ten times above the rest.
SELECT SUBSTRING(`Date`, 1, 7)          AS month,
       ROUND(CAST(`Close` AS DOUBLE), 2) AS close
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
WHERE symbol = 'CGT.L' AND `Date` >= '2024-11-01' AND `Date` <= '2025-11-01'
ORDER BY CAST(`Close` AS DOUBLE);

In [0]:
%sql
-- Threshold test. A trust can genuinely fall 30% in a month, so a tight threshold
-- mistakes real crashes for corruption. Compare two settings on the same data.
WITH base AS (
  SELECT symbol,
         SUBSTRING(`Date`, 1, 7)    AS month_key,
         CAST(`Close` AS DOUBLE)    AS close
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  WHERE CAST(`Close` AS DOUBLE) > 0
),
with_median AS (
  SELECT *,
         PERCENTILE_APPROX(close, 0.5) OVER (
           PARTITION BY symbol ORDER BY month_key
           ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING
         ) AS local_median
  FROM base
),
scored AS (
  SELECT *, close / local_median AS ratio
  FROM with_median
  WHERE local_median > 0 AND month_key >= '2011-09'
)
SELECT COUNT(*)                                                       AS rows_in_window,
       SUM(CASE WHEN ratio > 1.25 OR ratio < 0.8 THEN 1 ELSE 0 END)   AS flagged_tight,
       SUM(CASE WHEN ratio > 2    OR ratio < 0.5 THEN 1 ELSE 0 END)   AS flagged_wide,
       SUM(CASE WHEN month_key = '2020-03'
                 AND (ratio > 1.25 OR ratio < 0.8) THEN 1 ELSE 0 END) AS covid_flagged_tight,
       SUM(CASE WHEN month_key = '2020-03'
                 AND (ratio > 2 OR ratio < 0.5) THEN 1 ELSE 0 END)    AS covid_flagged_wide
FROM scored;

**The threshold matters more than the method.**

| Threshold | Rows flagged | Flagged in March 2020 |
|---|---|---|
| 1.25x / 0.8x | 532 | **22** — the COVID crash, wrongly caught |
| **2x / 0.5x** | **391** (2.43%) | **0** |

A trust can plausibly lose a third of its value in a month. It cannot plausibly gain 900%.
The wider threshold separates real market movement from unit errors cleanly.

In [0]:
%sql
-- Who is affected, how badly, and by what factor. The factor is what decides whether a
-- row could be rescaled or only dropped -- a clean 10x is a unit error, 4.07x is not.
WITH base AS (
  SELECT symbol,
         SUBSTRING(`Date`, 1, 7) AS month_key,
         CAST(`Close` AS DOUBLE) AS close
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
  WHERE CAST(`Close` AS DOUBLE) > 0
),
with_median AS (
  SELECT *,
         PERCENTILE_APPROX(close, 0.5) OVER (
           PARTITION BY symbol ORDER BY month_key
           ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING
         ) AS local_median
  FROM base
),
scored AS (
  SELECT *, close / local_median AS ratio
  FROM with_median
  WHERE local_median > 0 AND month_key >= '2011-09'
)
SELECT symbol,
       COUNT(*)                                                  AS months,
       SUM(CASE WHEN ratio > 2 OR ratio < 0.5 THEN 1 ELSE 0 END) AS flagged,
       ROUND(100.0 * SUM(CASE WHEN ratio > 2 OR ratio < 0.5 THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                      AS pct_flagged,
       ROUND(PERCENTILE_APPROX(CASE WHEN ratio > 2 OR ratio < 0.5 THEN ratio END, 0.5), 3)
                                                                 AS typical_factor
FROM scored
GROUP BY symbol
HAVING flagged > 0
ORDER BY flagged DESC;

**36 symbols affected, 391 rows, 2.43% of the window.**

Two things this table shows that matter to Silver:

1. **The factors are not uniform.** `CGT` is a clean **10x** — plainly a pence-versus-
   pounds unit error. But `FCIT` is 4.07x, `BRSC` 5.25x, `PIN` 1.53x, `INOV` 0.02x and
   `CLDN` around 1446x. There is no single factor to divide by.
2. **The bad rows are always a minority within a symbol** — the worst (`CLDN`, `BRSC`,
   `CGT`, `FCIT`) are about 29%. That is why the median holds up as a yardstick.

Dropping the flagged **rows** costs 2.43% of the window. Dropping the affected **trusts**
would cost 36 of 96, over a third of the study. **Open for Silver** — this notebook takes
no view on which.

## 7. Is `Adj_Close` usable?

In [0]:
%sql
-- Adj_Close is meant to fold dividends back in, so at a symbol's OLDEST bar it should sit
-- well below Close. If the ratio is ~1, no adjustment was applied.
WITH first_bar AS (
  SELECT symbol,
         CAST(`Close` AS DOUBLE)    AS close,
         CAST(Adj_Close AS DOUBLE)  AS adj_close,
         ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY `Date`) AS rn
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
)
SELECT CASE WHEN adj_close / close > 0.995 THEN 'no adjustment at all'
            WHEN adj_close / close > 0.95  THEN 'under 5 percent'
            ELSE 'a real adjustment' END AS adjustment,
       COUNT(*) AS symbols
FROM first_bar
WHERE rn = 1 AND close > 0
GROUP BY adjustment
ORDER BY symbols DESC;

In [0]:
%sql
-- HFEL is the clearest case: a high-yield trust whose dividends Yahoo never applied.
SELECT COUNT(*)                                  AS dividend_months,
       ROUND(SUM(CAST(Dividends AS DOUBLE)), 2)  AS total_dividends_gbp_pence
FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
WHERE symbol = 'HFEL.L'
  AND CAST(Dividends AS DOUBLE) > 0
  AND `Date` >= '2016-09-01';

**`Adj_Close` is unusable for UK trusts: 97 of 100 show under 5% adjustment.**

`HFEL` proves it — roughly 40 dividends totalling about 232p on a share that started the
decade around 359p, and Yahoo adjusted the series by under 1%. Its ten-year figures are a
price return of **−25.8%** against a total return of **+39.0%**.

The dividend *amounts* are present and correct; only Yahoo's adjustment is missing. So
total return has to be built from `Close` + `Dividends`. **Settled — never use
`Adj_Close` for the trusts.** See `profile_index_prices` for the contrast: it works fine
for SPY.

---

## Findings

| # | Finding | Status |
|---|---|---|
| 3.1 | Month key is `SUBSTRING(Date,1,7)`. All bars on day 01, 0 collisions. | **settled** |
| 3.2 | `Close` is already split-adjusted — 50 splits, `BNKR` continuous through 10-for-1. | **settled** |
| 3.3 | `Adj_Close` applies no dividends for UK trusts (97 of 100 under 5%). Build total return from `Close` + `Dividends`. | **settled** |
| 3.4 | 391 rows (2.43%) across 36 symbols sit over 2x from their neighbourhood median. Factors vary: `CGT` 10x, `FCIT` 4.07x, `CLDN` ~1446x. | **open** |
| 3.5 | Detection threshold must be 2x, not 1.25x — the tighter setting flags 22 trusts in March 2020, which is the COVID crash. | **settled** |
| 3.6 | One zero price: `PCFT.L` 2019-11-01, the same row the CSV carries. | **open** |
| 3.7 | The current month is a partial month for 97 symbols. | **open** |
| 3.8 | 4 stubs under 3 years: `ADIG`, `BSIF`, `EOT`, `MNTN`. | **open** |
| 3.9 | History depth: 75 at 15y+, 10 at 10–15y, 10 at 5–10y, 1 at 3–5y. These are the horizon denominators. | **settled** |

Everything marked **open** is a decision for the Silver spec. This notebook measures; it
does not choose.